Copyright: Vrije Universiteit Amsterdam, Faculty of Humanities, CLTL

# Lab 2.2: Basic Language Analysis

In the HLT course, you learned how to perform natural language processing steps using the libraries *nltk* and *spacy* in [Lab 1](https://github.com/cltl/ma-hlt-labs/tree/master/lab1.toolkits). Now is a good time to refresh your memory.  
Spacy and NLTK are only available for a few languages. In this lab, we will work with *stanza* which is available for more than 60 languages. Take a look at the [documentation](https://stanfordnlp.github.io/stanza/) for details. Stanza is optimized for accuracy and not for speed, so processing takes longer than with spacy. 

You are free to choose any of the libraries for your assignments. Just make sure to document your selection. 

In [1]:
import pandas as pd
import stanza

# Read in the data 
language = "fr"
article_file = "../data/veganism_overview_" + language +".tsv"
content = pd.read_csv(article_file, sep="\t", header = 0, keep_default_na=False)

# # Prepare the nlp pipeline
# stanza.download(language)
nlp = stanza.Pipeline(language)


2024-11-15 17:20:40 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2024-11-15 17:20:40 INFO: Downloaded file to C:\Users\NIKHIL\stanza_resources\resources.json
2024-11-15 17:20:43 INFO: Loading these models for language: fr (French):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |
| ner       | wikiner           |

2024-11-15 17:20:43 INFO: Using device: cpu
2024-11-15 17:20:43 INFO: Loading: tokenize
E:\Software\Anaconda\envs\LaD\lib\site-packages\stanza\models\tokenization\trainer.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default val

We have loaded the pipeline and can start processing our content. In the example, we only use the first article. **Once you understand how it works, modify the code to iterate through all articles and save the result.** 

In [2]:
# Process the first article
current_article = content["Text"][0]
processed_article = nlp(current_article)

## 1. Tokenization

The stanza pipeline detects sentence boundaries and segments the texts into tokens. **Analyze the output and check the tokenization quality.** 

In [3]:
sentences = processed_article.sentences

# For the example, I only look at the first three sentences. Make sure to change this. 
for sentence in sentences[0:4]:
    print("Sentence: ", sentence.text)
    print("Tokens: ")
    for token in sentence.tokens:
        print(token.id[0], token.text)
    print()

Sentence:  Les véganeries vont de plus en plus loin.
Tokens: 
1 Les
2 véganeries
3 vont
4 de
5 plus
6 en
7 plus
8 loin
9 .

Sentence:  Connaissez-vous le "faux mage" à base de lait de noix de cajou ?
Tokens: 
1 Connaissez
2 -vous
3 le
4 "
5 faux
6 mage
7 "
8 à
9 base
10 de
11 lait
12 de
13 noix
14 de
15 cajou
16 ?

Sentence:  ... Article sans intérêt.
Tokens: 
1 ...
2 Article
3 sans
4 intérêt
5 .

Sentence:  Oui le faux-mage existe et alors ?
Tokens: 
1 Oui
2 le
3 faux-mage
4 existe
5 et
6 alors
7 ?



## 2. Token Frequencies

Now, let's count the token frequencies in this article. If necessary, go back to [chapter 10](https://github.com/cltl/python-for-text-analysis/blob/master/Chapters/Chapter%2010%20-%20Dictionaries.ipynb) of the python course and refresh your skills on how to count with dictionaries and counters.  

In [4]:
from collections import Counter

token_frequencies = Counter()

for sentence in sentences:
    all_tokens =[token.text for token in sentence.tokens]
    token_frequencies.update(all_tokens)
    
# Print the most common tokens
print(token_frequencies.most_common(40))

[('de', 34), (',', 31), ('la', 18), ('.', 17), ('...', 15), ('des', 15), ('?', 13), ('et', 13), ('en', 12), ('à', 11), ('pas', 10), ('le', 9), ("l'", 9), ('produits', 9), ('pour', 9), ('les', 9), ("d'", 7), ('est', 7), ('a', 6), ('vous', 6), ('qui', 6), ('ne', 6), ('ils', 6), ('plus', 5), ('lait', 5), ('cajou', 5), ('alors', 5), ('ça', 5), ('ou', 5), ('notre', 5), ('viande', 5), ('nous', 5), ('mais', 5), ('"', 4), ('noix', 4), ('y', 4), ('soja', 4), ('un', 4), ('une', 4), ("n'", 4)]


The type-token ratio is an indicator for lexical variation. **Think about example texts with very high or very low type-token ratio.**

In [5]:
# Type-token ratio
num_types = len(token_frequencies.keys())
num_tokens = sum(token_frequencies.values())

tt_ratio = num_types/num_tokens

print(num_types, num_tokens)

# Print the type token ratio with 4 decimals
print("%.4f" % tt_ratio)

333 739
0.4506


## 3. Saving as pickle

We can save complex Python objects like dictionaries (or like the processed articles) in a *pickle* file. This can be convenient if you are running a processing step that takes a lot of time and you want to do it only once. You can then save the intermediate output in a pickle-file and load it directly when you continue working on it. 

Note that pickle files can also be used to hide harmful code. So make sure to only open pickle files if you know who created them and what they contain. More information can be found in this [tutorial](https://www.datacamp.com/community/tutorials/pickle-python-tutorial).

When opening files, *w* stands for write, *r* for read and *b* indicates that the file is binary.

In [6]:
import pickle
frequency_file = "../data/processed_data/tokenfrequencies_article1.pkl"
pickle.dump(token_frequencies, open(frequency_file, "wb"))

stanza_objects_file = "../data/processed_data/nlp_article1.pkl"
pickle.dump(processed_article, open(stanza_objects_file, "wb"))

# You can then later load the file like this: 
# loaded_frequencies = pickle.load(open(frequencyfile, "rb")) 

## 4. Token frequencies of all articles

In the example, we only used the first article. **Once you understand how it works, modify the code to iterate through all articles and save the result in "../data/processed_data/tokenfrequencies.pkl".** 

Stanza processing takes quite long, so you can try it directly with your own dataset, if you prefer. Do not save all stanza objects, the file will get quite big. 

In [7]:
# Initialize variables
token_frequencies = Counter()
text = content["Text"]
count = 1
# Iterate through all articles
for current_article in text:
    print(f"Processing {count}/{len(text)}")
    count+=1
    # Skip empty articles
    if len(current_article) == 0:
        continue
    # Process the article with the stanza pipeline
    processed_article = nlp(current_article)
    sentences = processed_article.sentences
    # Iterate through all sentences of the article
    for sentence in sentences:
        all_tokens =[token.text for token in sentence.tokens]
        # Add the tokens to a counter
        token_frequencies.update(all_tokens)

num_types = len(token_frequencies.keys())
num_tokens = sum(token_frequencies.values())

tt_ratio = num_types/num_tokens

Processing 1/100
Processing 2/100
Processing 3/100
Processing 4/100
Processing 5/100
Processing 6/100
Processing 7/100
Processing 8/100
Processing 9/100
Processing 10/100
Processing 11/100
Processing 12/100
Processing 13/100
Processing 14/100
Processing 15/100
Processing 16/100
Processing 17/100
Processing 18/100
Processing 19/100
Processing 20/100
Processing 21/100
Processing 22/100
Processing 23/100
Processing 24/100
Processing 25/100
Processing 26/100
Processing 27/100
Processing 28/100
Processing 29/100
Processing 30/100
Processing 31/100
Processing 32/100
Processing 33/100
Processing 34/100
Processing 35/100
Processing 36/100
Processing 37/100
Processing 38/100
Processing 39/100
Processing 40/100
Processing 41/100
Processing 42/100
Processing 43/100
Processing 44/100
Processing 45/100
Processing 46/100
Processing 47/100
Processing 48/100
Processing 49/100
Processing 50/100
Processing 51/100
Processing 52/100
Processing 53/100
Processing 54/100
Processing 55/100
Processing 56/100
P

In [8]:
print(num_types,num_tokens)
print("%.4f" % tt_ratio)
print(token_frequencies.most_common(50))

11272 80656
0.1398
[(',', 4437), ('de', 3678), ('.', 2650), ('la', 1607), ('les', 1455), ('et', 1379), ('le', 1253), ('à', 1161), ('des', 1140), ('est', 926), ('en', 910), ('l’', 844), ('un', 837), ('d’', 736), ('que', 734), ('une', 704), ('pour', 615), ('qui', 591), ('«', 572), ('du', 549), ('»', 543), ('dans', 513), ('pas', 508), ('a', 479), ('plus', 469), ("l'", 451), ('ne', 412), ('par', 388), ('sur', 375), (')', 373), ('(', 365), ('ou', 364), (':', 356), ("d'", 331), ('animaux', 328), ('ce', 318), ('il', 312), ('sont', 303), ('se', 289), ('"', 288), ('au', 278), ('qu’', 267), ('viande', 247), ('?', 236), ('comme', 223), ('produits', 215), ('animale', 202), ('vegan', 196), ('avec', 196), ('Le', 190)]


In [9]:
# Save the token frequencies as a pickle file
frequency_file = "../data/processed_data/tokenfrequencies.pkl"
pickle.dump(token_frequencies, open(frequency_file, "wb"))



## 5. More linguistic processing

The pipeline provides a large amount of additional information. Let's check the representation of the sentence. **Try to understand which information is represented by the attributes *lemma*, *upos*, *feats*, *heads*, *deprel* and *ner*.** We will learn more about this in the next week, but you can already collect some statistics over the information.   

In [10]:
print(sentences[0])

[
  {
    "id": 1,
    "text": "Devenir",
    "lemma": "devenir",
    "upos": "VERB",
    "feats": "VerbForm=Inf",
    "head": 24,
    "deprel": "advcl",
    "start_char": 0,
    "end_char": 7,
    "ner": "O",
    "multi_ner": [
      "O"
    ]
  },
  {
    "id": 2,
    "text": "végan",
    "lemma": "végan",
    "upos": "ADJ",
    "feats": "Gender=Masc|Number=Sing",
    "head": 1,
    "deprel": "xcomp",
    "start_char": 8,
    "end_char": 13,
    "ner": "O",
    "multi_ner": [
      "O"
    ]
  },
  {
    "id": 3,
    "text": "pendant",
    "lemma": "pendant",
    "upos": "ADP",
    "head": 5,
    "deprel": "case",
    "start_char": 14,
    "end_char": 21,
    "ner": "O",
    "multi_ner": [
      "O"
    ]
  },
  {
    "id": 4,
    "text": "un",
    "lemma": "un",
    "upos": "NUM",
    "feats": "Number=Sing",
    "head": 5,
    "deprel": "nummod",
    "start_char": 22,
    "end_char": 24,
    "ner": "O",
    "multi_ner": [
      "O"
    ]
  },
  {
    "id": 5,
    "text": "mois",
   